In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.carga import cargar_datos_cen
from src.nucleo_poo import FiltradorCentrales, TransformadorAnchoLargo, ConstructorVariablesDerivadas, Pipeline
from src.validacion import validar_dataset_procesado
from src.agregacion_temporal import AgregacionHoraria, AgregacionDiaria, AgregacionMensual, AgregacionPorDiaSemana, CaracterizadorTemporal, comparar_centrales
import pandas as pd

In [2]:
centrales = ["TER CMPC LAJA", "TER CMPC PACIFICO", "TER CMPC SANTA FE"]
columnas_base = ["Año", "Mes", "Llave", "Central", "Coordinado", "Grupo reporte", "Tipo", "Subtipo", "Fecha"]
columnas_hora = [f"Hora {i}" for i in range(1, 25)]

datos_crudos = cargar_datos_cen("../data/raw/generacion_real_cen_ene_ago_2026.csv")
datos_crudos.shape

(359891, 33)

### Núcleo algorítmico: Pipeline orientado a objetos

Los tres pasos del pipeline de F2 (filtrar centrales, transformar ancho→largo, construir variables derivadas) se reorganizan como clases derivadas de una clase base Transformador, compuestas en una clase Pipeline. Esto no cambia la lógica de F2: la reutiliza (carga.py, transformacion.py) bajo un contrato orientado a objetos común, con alta cohesión (cada clase tiene una sola responsabilidad) y bajo acoplamiento (el Pipeline no conoce el tipo concreto de cada paso).

In [3]:
pipeline = Pipeline([
    FiltradorCentrales(centrales),
    TransformadorAnchoLargo(columnas_base, columnas_hora),
    ConstructorVariablesDerivadas(),
])

datos_finales = pipeline.ejecutar(datos_crudos)
datos_finales.shape

(17496, 13)

### Sobre recursividad

El pipeline tiene tres pasos fijos y conocidos (filtrar, transformar, construir variables), no un problema de profundidad variable. Por eso se optó por división funcional en clases en vez de recursividad — alternativa que el descriptor de la rúbrica admite explícitamente.

### Verificación de equivalencia funcional

Se contrasta el resultado del Pipeline con el dataset procesado oficial de F2, mediante pd.testing.assert_frame_equal. La coincidencia exacta constituye evidencia objetiva de que la reorganización en clases preservó la semántica del pipeline original.

In [4]:
oficial = pd.read_csv("../data/processed/dataset_cen_centrales_cmpc_ene_ago_2026.csv")

a = datos_finales.sort_values("ID_Observacion").reset_index(drop=True)
b = oficial.sort_values("ID_Observacion").reset_index(drop=True)

pd.testing.assert_frame_equal(a, b, check_dtype=False)
print("Equivalencia funcional confirmada con F2 ✔")

Equivalencia funcional confirmada con F2 ✔


In [5]:
for columna in a.columns:
    try:
        pd.testing.assert_series_equal(a[columna], b[columna], check_dtype=False)
        print(f"{columna}: OK")
    except AssertionError as e:
        print(f"{columna}: DIFERENCIA")
        print(e)
        print()

ID_Observacion: OK
Año: OK
Mes: OK
Llave: OK
Central: OK
Coordinado: OK
Grupo_Reporte: OK
Tipo: OK
Subtipo: OK
Fecha: OK
Dia_Semana: OK
Hora: OK
Generacion_MWh: OK


### Eficiencia y optimización

Se comparan dos implementaciones de la transformación ancho→largo: la vectorizada (pandas.melt, usada en producción) y una iterativa (iterrows), verificando primero su equivalencia funcional y luego midiendo tiempo (timeit) y memoria (tracemalloc) sobre las 729 filas reales del proyecto.

In [7]:
import re
import timeit
import tracemalloc

def transformar_ancho_largo_iterativo(datos, columnas_base, columnas_hora):
    """Misma transformación, pero con un bucle manual en vez de melt."""
    filas = []
    for _, fila in datos.iterrows():
        base = {col: fila[col] for col in columnas_base}
        for columna_hora in columnas_hora:
            numero_hora = int(re.search(r"(\d+)", columna_hora).group(1))
            nueva_fila = dict(base)
            nueva_fila["Hora"] = numero_hora
            nueva_fila["Generacion_MWh"] = pd.to_numeric(fila[columna_hora], errors="raise")
            filas.append(nueva_fila)
    resultado = pd.DataFrame(filas)
    return resultado[columnas_base + ["Hora", "Generacion_MWh"]]

# Datos reales filtrados a las 3 centrales (729 filas anchas)
datos_729 = FiltradorCentrales(centrales).ajustar(datos_crudos).transformar(datos_crudos)

# 1. Verificar equivalencia
from src.transformacion import transformar_ancho_largo
a = transformar_ancho_largo(datos_729, columnas_base, columnas_hora).sort_values(columnas_base+["Hora"]).reset_index(drop=True)
b = transformar_ancho_largo_iterativo(datos_729, columnas_base, columnas_hora).sort_values(columnas_base+["Hora"]).reset_index(drop=True)
pd.testing.assert_frame_equal(a, b, check_dtype=False)
print("Ambas formas producen el mismo resultado.\n")

# 2. Medir tiempo
rep = 30
t_v = timeit.timeit(lambda: transformar_ancho_largo(datos_729, columnas_base, columnas_hora), number=rep) / rep
t_i = timeit.timeit(lambda: transformar_ancho_largo_iterativo(datos_729, columnas_base, columnas_hora), number=rep) / rep
print(f"melt (vectorizado): {t_v:.6f} s")
print(f"iterrows (bucle)  : {t_i:.6f} s")
print(f"El vectorizado es {t_i/t_v:,.1f} veces más rápido\n")

# 3. Medir memoria
tracemalloc.start(); transformar_ancho_largo(datos_729, columnas_base, columnas_hora); _, p_v = tracemalloc.get_traced_memory(); tracemalloc.stop()
tracemalloc.start(); transformar_ancho_largo_iterativo(datos_729, columnas_base, columnas_hora); _, p_i = tracemalloc.get_traced_memory(); tracemalloc.stop()
print(f"Memoria melt    : {p_v/1024/1024:.4f} MB")
print(f"Memoria iterrows: {p_i/1024/1024:.4f} MB")

Ambas formas producen el mismo resultado.

melt (vectorizado): 0.031491 s
iterrows (bucle)  : 0.155673 s
El vectorizado es 4.9 veces más rápido

Memoria melt    : 4.3982 MB
Memoria iterrows: 11.1392 MB


### Patrón de diseño: Strategy

Antes, calcular la generación por hora, día y mes habría requerido tres bloques de código repetidos. Con Strategy, cada nivel de agregación es una clase intercambiable (AgregacionHoraria, AgregacionDiaria, AgregacionMensual, AgregacionPorDiaSemana) que implementa el mismo contrato, y CaracterizadorTemporal las aplica sin conocer cuál es. Sin este patrón, agregar un nivel nuevo habría exigido modificar código existente en vez de solo agregar una clase.

In [8]:
resultado_horario = comparar_centrales(datos_finales, AgregacionHoraria())
resultado_mensual = comparar_centrales(datos_finales, AgregacionMensual())

print("--- Perfil horario promedio por central (primeras 6 horas) ---")
print(resultado_horario.head(6).round(2))
print("\n--- Generación total mensual por central ---")
print(resultado_mensual.round(1))

--- Perfil horario promedio por central (primeras 6 horas) ---
Central  TER CMPC LAJA  TER CMPC PACIFICO  TER CMPC SANTA FE
Hora                                                        
1                 3.43              17.00               4.99
2                 3.44              16.81               4.87
3                 3.56              16.03               4.81
4                 3.59              16.46               4.98
5                 3.54              16.72               5.19
6                 3.29              16.48               5.12

--- Generación total mensual por central ---
Central  TER CMPC LAJA  TER CMPC PACIFICO  TER CMPC SANTA FE
Mes                                                         
Abr             1560.2            12673.3             5712.3
Ago              298.0             9395.9             1333.0
Ene             5662.2            12840.9             1905.3
Feb             4540.3             9516.6             2952.1
Jul              448.6             89